 # Data Format Conversion for DR4SR



 This notebook converts the processed recommendation data into the format required by the DR4SR model.

In [ ]:
import os

import pandas as pd
import torch
from recbole.utils import init_seed
from tqdm import tqdm

# Initialize random seed
init_seed(2025, True)


 ## Configuration

In [ ]:
# Path configuration
INPUT_DIR = "BEST"
OUTPUT_DIR = "path/to/DR4SR/dataset"
OUTPUT_PATH = "dom1-new"

# Model parameters
MAX_SEQ_LENGTH = 128
PAD_TOKEN = 0  # Padding token index

# Input files
# .new-2025-CE.
TRAIN_INPUT_FILE = f"{INPUT_DIR}.2025.dom1.train.inter"
VALID_INPUT_FILE = f"{INPUT_DIR}.dom1.valid.inter"
TEST_INPUT_FILE = f"{INPUT_DIR}.dom1.test.inter"

# Create output directory structure
os.makedirs(os.path.join(OUTPUT_DIR, OUTPUT_PATH, OUTPUT_PATH), exist_ok=True)


 ## Data Loading Utilities

In [ ]:
def load_interaction_file(file_path: str) -> pd.DataFrame:
    """Load interaction file into DataFrame with proper column types."""
    return pd.read_csv(
        file_path,
        sep="\t",
        header=0,
        names=["user_id", "item_id_list", "target_item"],
        dtype={"user_id": str, "item_id_list": str, "target_item": str},
    )


# Load all data files
print("Loading data files...")
train_df = load_interaction_file(os.path.join(INPUT_DIR, TRAIN_INPUT_FILE))
valid_df = load_interaction_file(os.path.join(INPUT_DIR, VALID_INPUT_FILE))
test_df = load_interaction_file(os.path.join(INPUT_DIR, TEST_INPUT_FILE))


 ## ID Mapping Generation

In [ ]:
def create_id_mappings(train_df: pd.DataFrame, valid_df: pd.DataFrame, test_df: pd.DataFrame) -> tuple:
    """Create mappings between original IDs and numerical indices."""
    print("Creating ID mappings...")

    # Get all unique users
    all_users = pd.concat([train_df["user_id"], valid_df["user_id"], test_df["user_id"]]).unique()

    # Get all unique items
    all_items = set()
    for df in [train_df, valid_df, test_df]:
        for _, row in df.iterrows():
            items = row["item_id_list"].split() + [str(row["target_item"])]
            all_items.update(items)
    all_items = list(all_items)

    # Create mappings (1-based indexing with 0 as PAD)
    user_mapping = {orig: idx + 1 for idx, orig in enumerate(all_users)}
    item_mapping = {orig: idx + 1 for idx, orig in enumerate(all_items)}

    num_users = len(user_mapping) + 1  # +1 for PAD
    num_items = len(item_mapping) + 1  # +1 for PAD

    print(f"Total users: {num_users - 1} (with PAD)")
    print(f"Total items: {num_items - 1} (with PAD)")

    return user_mapping, item_mapping, num_users, num_items


# Create mappings
user_mapping, item_mapping, num_users, num_items = create_id_mappings(train_df, valid_df, test_df)


 ## Data Transformation

In [ ]:
def apply_id_mappings(df: pd.DataFrame, user_map: dict, item_map: dict) -> pd.DataFrame:
    """Apply ID mappings to convert original IDs to numerical indices."""
    df = df.copy()
    df["user_id"] = df["user_id"].map(user_map)
    df["item_id_list"] = df["item_id_list"].apply(lambda x: [item_map[i] for i in x.split() if i in item_map])
    df["target_item"] = df["target_item"].map(item_map)
    return df


# Apply mappings to all dataframes
train_df = apply_id_mappings(train_df, user_mapping, item_mapping)
valid_df = apply_id_mappings(valid_df, user_mapping, item_mapping)
test_df = apply_id_mappings(test_df, user_mapping, item_mapping)


 ## Sequence Pattern Generation

In [ ]:
def generate_sequence_patterns(df: pd.DataFrame) -> list:
    """Generate sequence patterns for training data."""
    return [row["item_id_list"] + [row["target_item"]] for _, row in df.iterrows()]


# Generate and save sequence patterns
seq_patterns = generate_sequence_patterns(train_df)
pattern_output_path = os.path.join(OUTPUT_DIR, OUTPUT_PATH, OUTPUT_PATH, "seq2pat_data.pth")
torch.save(seq_patterns, pattern_output_path)


 ## Sample Generation

In [ ]:
def process_sequence(sequence: list, max_len: int = MAX_SEQ_LENGTH) -> tuple:
    """Truncate or pad sequence to fixed length."""
    seq_len = len(sequence)
    if seq_len > max_len:
        return sequence[-max_len:], max_len
    return sequence + [PAD_TOKEN] * (max_len - seq_len), seq_len


def create_model_samples(df: pd.DataFrame, split_type: str) -> list:
    """Create samples in format required by the model."""
    samples = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Creating {split_type} samples"):
        user_id = row["user_id"]
        full_sequence = row["item_id_list"] + [row["target_item"]]

        if split_type in ["valid", "test"]:
            history, seq_len = process_sequence(full_sequence[:-1])
            target = full_sequence[-1]
            label = 1
            domain_id = [0] * MAX_SEQ_LENGTH  # Single domain (0)
            samples.append([user_id, history, target, seq_len, label, domain_id, history])

        elif split_type == "train":
            history, seq_len = process_sequence(full_sequence[:-1])
            target, _ = process_sequence(full_sequence[1:])
            label = [1] * seq_len + [PAD_TOKEN] * (MAX_SEQ_LENGTH - seq_len)
            domain_id = [0] * MAX_SEQ_LENGTH  # Single domain (0)
            samples.append([user_id, history, target, seq_len, label, domain_id])

    return samples


# Generate and save samples
print("\nGenerating model samples...")
train_samples = create_model_samples(train_df, "train")
valid_samples = create_model_samples(valid_df, "valid")
test_samples = create_model_samples(test_df, "test")

torch.save(train_samples, os.path.join(OUTPUT_DIR, OUTPUT_PATH, OUTPUT_PATH, "train.pth"))
torch.save(valid_samples, os.path.join(OUTPUT_DIR, OUTPUT_PATH, OUTPUT_PATH, "val.pth"))
torch.save(test_samples, os.path.join(OUTPUT_DIR, OUTPUT_PATH, OUTPUT_PATH, "test.pth"))


In [ ]:
print(train_samples[0][0])
print(train_samples[0][1])
print(train_samples[0][2])
print(train_samples[0][3])
print(train_samples[0][4])
print(train_samples[0][5])
print("==================")
print(valid_samples[0][0])
print(valid_samples[0][1])
print(valid_samples[0][2])
print(valid_samples[0][3])
print(valid_samples[0][4])
print(valid_samples[0][5])
print(valid_samples[0][6])

 ## Full Dataset Export

In [ ]:
def create_full_interaction_dataset(
    train_df: pd.DataFrame, valid_df: pd.DataFrame, test_df: pd.DataFrame
) -> pd.DataFrame:
    """Create a unified interaction dataset for evaluation."""
    print("\nCreating full interaction dataset...")

    user_list, item_list = [], []
    full_df = pd.concat([train_df, valid_df, test_df])

    for _, row in tqdm(full_df.iterrows(), total=len(full_df)):
        user_id = row["user_id"]
        items = row["item_id_list"] + [row["target_item"]]

        # Add user-item pairs
        user_list.extend([user_id] * len(items))
        item_list.extend(items)

    # Create DataFrame with required columns
    return pd.DataFrame(
        {
            "user_id": user_list,
            "item_id": item_list,
            "rating": [5] * len(user_list),  # Default rating
            "timestamp": range(len(user_list)),  # Sequential timestamps
            "domain": [0] * len(user_list),  # Single domain
        }
    )


# Create and save full dataset
full_dataset = create_full_interaction_dataset(train_df, valid_df, test_df)
full_dataset.to_csv(os.path.join(OUTPUT_DIR, OUTPUT_PATH, OUTPUT_PATH, "inter.csv"), index=False)

# Print dataset statistics
stat_file = os.path.join(OUTPUT_DIR, OUTPUT_PATH, "dataset_statistics.txt")
with open(stat_file, "w") as f:
    f.write("\nDataset statistics:\n")
    f.write(f"Unique users: {full_dataset['user_id'].nunique()}\n")
    f.write(f"Unique items: {full_dataset['item_id'].nunique()}\n")
    f.write(f"Total interactions: {len(full_dataset)}\n")
print("\nDataset statistics:")
print(f"Unique users: {full_dataset['user_id'].nunique()}")
print(f"Unique items: {full_dataset['item_id'].nunique()}")
print(f"Total interactions: {len(full_dataset)}")
